# KOCOH Baseline 최종본

`KOCOH_v.2.csv` 기준으로 실행하는 최종 노트북입니다.

포함:
- KOCOH 데이터 전처리
- 팀원 전달용 `team_train/valid/test.csv` 생성
- Baseline용 `baseline_train/valid/test.csv` 생성
- BERT 댓글 단독 Baseline
- RoBERTa 댓글 단독 Baseline
- `baseline_result.csv` 저장

모델 파일은 저장하지 않습니다. Docker 저장공간 오류를 피하기 위한 버전입니다.

In [ ]:
!pip install pandas scikit-learn transformers torch -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## 1. 라이브러리 및 기본 설정

In [6]:
import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
MAX_LENGTH = 128
BATCH_SIZE = 4
EPOCHS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)
print("EPOCHS:", EPOCHS)
print("BATCH_SIZE:", BATCH_SIZE)

사용 장치: cpu
EPOCHS: 5
BATCH_SIZE: 4


## 2. KOCOH 데이터 불러오기

In [7]:
possible_paths = ["KOCOH_v.2.csv", "KOCOH_v2.csv", "KOCOH_v.2", "KOCOH_v2"]
DATA_PATH = None

for path in possible_paths:
    if os.path.exists(path):
        DATA_PATH = path
        break

if DATA_PATH is None:
    print("현재 폴더 파일 목록:")
    print(os.listdir("."))
    raise FileNotFoundError("KOCOH_v.2.csv 파일을 찾지 못했습니다.")

print("사용할 데이터 파일:", DATA_PATH)

df = pd.read_csv(DATA_PATH)

print("데이터 크기:", df.shape)
display(df.head())
print("원본 컬럼명:", df.columns.tolist())

사용할 데이터 파일: KOCOH_v.2.csv
데이터 크기: (3000, 18)


,Index,Set,Type,Date,Source,Link,Title,Context,Comment,Hate speech,Counter speech,Gender,Disability,Race/Nationality,Region (Korea),Age,Profanity,Ikiyano-style
0,1,1,1,2024-06-02,디시인사이드,https://gall.dcinside.com/board/view/?id=dcbes...,싱글벙글 이제부터 여성 안 뽑는 '여혐업체' 박제한다....jpg,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,미래가 밝은 회사이니 투자하라는 뜻에서 샤라웃 해주는 거임,1,0,1,0,0,0,0,0,0
1,2,1,1,2024-06-02,디시인사이드,https://gall.dcinside.com/board/view/?id=dcbes...,싱글벙글 이제부터 여성 안 뽑는 '여혐업체' 박제한다....jpg,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,쉐보레 차 좋은데엔 이유가 있었노,1,0,1,0,0,0,0,0,1
2,3,1,1,2024-06-02,디시인사이드,https://gall.dcinside.com/board/view/?id=dcbes...,싱글벙글 이제부터 여성 안 뽑는 '여혐업체' 박제한다....jpg,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,풀매수 간다,1,0,1,0,0,0,0,0,0
3,4,1,1,2024-06-02,디시인사이드,https://gall.dcinside.com/board/view/?id=dcbes...,싱글벙글 이제부터 여성 안 뽑는 '여혐업체' 박제한다....jpg,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,옳게된 기업,1,0,1,0,0,0,0,0,0
4,5,1,1,2024-06-02,디시인사이드,https://gall.dcinside.com/board/view/?id=dcbes...,싱글벙글 이제부터 여성 안 뽑는 '여혐업체' 박제한다....jpg,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,그니까 저기에 주식투자 하면 된다는거지?,1,0,1,0,0,0,0,0,0


원본 컬럼명: ['Index', 'Set', 'Type', 'Date', 'Source', 'Link', 'Title', 'Context', 'Comment', 'Hate speech', 'Counter speech', 'Gender', 'Disability', 'Race/Nationality', 'Region (Korea)', 'Age', 'Profanity', 'Ikiyano-style']


## 3. 컬럼명 정리

In [8]:
df.columns = [str(col).strip().lower() for col in df.columns]

print("소문자 변환 후 컬럼명:")
print(df.columns.tolist())

if "hate speech" in df.columns:
    df = df.rename(columns={"hate speech": "label"})
elif "hate_speech" in df.columns:
    df = df.rename(columns={"hate_speech": "label"})
elif "hatespeech" in df.columns:
    df = df.rename(columns={"hatespeech": "label"})
elif "label" in df.columns:
    pass
else:
    raise ValueError("라벨 컬럼을 찾지 못했습니다.")

for col in ["context", "comment", "label"]:
    if col not in df.columns:
        raise ValueError(f"필수 컬럼이 없습니다: {col}")

if "type" not in df.columns:
    df["type"] = "unknown"

df = df[["context", "comment", "label", "type"]].copy()

df["context"] = df["context"].astype(str)
df["comment"] = df["comment"].astype(str)
df["label"] = df["label"].astype(int)
df["type"] = df["type"].astype(str)

display(df.head())

print("라벨 분포:")
print(df["label"].value_counts())

소문자 변환 후 컬럼명:
['index', 'set', 'type', 'date', 'source', 'link', 'title', 'context', 'comment', 'hate speech', 'counter speech', 'gender', 'disability', 'race/nationality', 'region (korea)', 'age', 'profanity', 'ikiyano-style']


,context,comment,label,type
0,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,미래가 밝은 회사이니 투자하라는 뜻에서 샤라웃 해주는 거임,1,1
1,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,쉐보레 차 좋은데엔 이유가 있었노,1,1
2,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,풀매수 간다,1,1
3,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,옳게된 기업,1,1
4,여성 고용이 미흡해 정부가 불이익을 준 기업의 이름이 공개되었다.,그니까 저기에 주식투자 하면 된다는거지?,1,1


라벨 분포:
label
0    2236
1     764
Name: count, dtype: int64


## 4. 결측치 및 중복 제거

In [9]:
print("결측치 개수:")
print(df.isnull().sum())

print("\n중복 데이터 개수:", df.duplicated().sum())

df = df.dropna(subset=["context", "comment", "label"])
df = df.drop_duplicates().reset_index(drop=True)

print("\n정리 후 데이터 크기:", df.shape)
print("\n정리 후 라벨 분포:")
print(df["label"].value_counts())

결측치 개수:
context    0
comment    0
label      0
type       0
dtype: int64

중복 데이터 개수: 6

정리 후 데이터 크기: (2994, 4)

정리 후 라벨 분포:
label
0    2230
1     764
Name: count, dtype: int64


## 5. Train / Validation / Test 분리

In [10]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df["label"]
)

print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test:", test_df.shape)

train: (2395, 4)
valid: (299, 4)
test: (300, 4)


## 6. 팀원 전달용 데이터 저장

In [11]:
team_cols = ["context", "comment", "label", "type"]

train_df[team_cols].to_csv("team_train.csv", index=False, encoding="utf-8-sig")
valid_df[team_cols].to_csv("team_valid.csv", index=False, encoding="utf-8-sig")
test_df[team_cols].to_csv("team_test.csv", index=False, encoding="utf-8-sig")

print("팀원 전달용 파일 저장 완료")
print("team_train.csv")
print("team_valid.csv")
print("team_test.csv")

팀원 전달용 파일 저장 완료
team_train.csv
team_valid.csv
team_test.csv


## 7. Baseline용 데이터 저장

In [13]:
baseline_cols = ["comment", "label"]

train_df[baseline_cols].to_csv("baseline_train.csv", index=False, encoding="utf-8-sig")
valid_df[baseline_cols].to_csv("baseline_valid.csv", index=False, encoding="utf-8-sig")
test_df[baseline_cols].to_csv("baseline_test.csv", index=False, encoding="utf-8-sig")

print("Baseline용 파일 저장 완료")
print("baseline_train.csv")
print("baseline_valid.csv")
print("baseline_test.csv")

Baseline용 파일 저장 완료
baseline_train.csv
baseline_valid.csv
baseline_test.csv


## 8. Dataset 클래스와 평가 함수

In [14]:
class CommentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.comments = dataframe["comment"].astype(str).tolist()
        self.labels = dataframe["label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.comments)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.comments[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


def evaluate_model(model, dataloader):
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="binary",
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "y_true": all_labels,
        "y_pred": all_preds
    }

## 9. Baseline 학습 함수

In [16]:
def train_baseline_model(model_name, output_name):
    print("=" * 70)
    print("학습 모델:", model_name)
    print("=" * 70)

    train_data = pd.read_csv("baseline_train.csv")
    valid_data = pd.read_csv("baseline_valid.csv")
    test_data = pd.read_csv("baseline_test.csv")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.to(device)

    train_dataset = CommentDataset(train_data, tokenizer, MAX_LENGTH)
    valid_dataset = CommentDataset(valid_data, tokenizer, MAX_LENGTH)
    test_dataset = CommentDataset(test_data, tokenizer, MAX_LENGTH)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # 클래스 불균형 보정
    label_counts = train_data["label"].value_counts().sort_index()
    total_count = len(train_data)

    weight_0 = total_count / (2 * label_counts[0])
    weight_1 = total_count / (2 * label_counts[1])

    class_weights = torch.tensor([weight_0, weight_1], dtype=torch.float).to(device)

    print("Class weights:", class_weights)

    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    for epoch in range(EPOCHS):
        model.train()
        total_train_loss = 0

        for step, batch in enumerate(train_loader, start=1):
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            loss = loss_fn(logits, labels)

            total_train_loss += loss.item()
            loss.backward()
            optimizer.step()

            if step % 50 == 0:
                print(f"Epoch {epoch + 1}/{EPOCHS} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}")

        avg_train_loss = total_train_loss / len(train_loader)
        valid_result = evaluate_model(model, valid_loader)

        print(f"\nEpoch {epoch + 1}/{EPOCHS} 결과")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Valid Accuracy: {valid_result['accuracy']:.4f}")
        print(f"Valid Precision: {valid_result['precision']:.4f}")
        print(f"Valid Recall: {valid_result['recall']:.4f}")
        print(f"Valid F1: {valid_result['f1']:.4f}")
        print("-" * 70)

    test_result = evaluate_model(model, test_loader)

    print("\n===== TEST RESULT =====")
    print(f"Accuracy : {test_result['accuracy']:.4f}")
    print(f"Precision: {test_result['precision']:.4f}")
    print(f"Recall   : {test_result['recall']:.4f}")
    print(f"F1-score : {test_result['f1']:.4f}")

    print("\n===== Classification Report =====")
    print(classification_report(test_result["y_true"], test_result["y_pred"], digits=4))

    print("\n===== Confusion Matrix =====")
    print(confusion_matrix(test_result["y_true"], test_result["y_pred"]))

    return {
        "model": output_name + "_class_weight",
        "accuracy": test_result["accuracy"],
        "precision": test_result["precision"],
        "recall": test_result["recall"],
        "f1": test_result["f1"]
    }

## 10. Baseline 1: BERT 기반 댓글 단독 분류 모델

In [15]:
bert_result = train_baseline_model(
    model_name="klue/bert-base",
    output_name="BERT_comment_only"
)

bert_result

NameError: name 'train_baseline_model' is not defined

## 11. Baseline 2: RoBERTa 기반 댓글 단독 분류 모델

In [ ]:
roberta_result = train_baseline_model(
    model_name="klue/roberta-base",
    output_name="RoBERTa_comment_only"
)

roberta_result

학습 모델: klue/roberta-base


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Class weights: tensor([0.6712, 1.9599])
Epoch 1/5 | Step 50/599 | Loss: 0.8682
Epoch 1/5 | Step 100/599 | Loss: 0.5385
Epoch 1/5 | Step 150/599 | Loss: 0.7740
Epoch 1/5 | Step 200/599 | Loss: 0.3630
Epoch 1/5 | Step 250/599 | Loss: 0.1628
Epoch 1/5 | Step 300/599 | Loss: 0.7835
Epoch 1/5 | Step 350/599 | Loss: 0.5676
Epoch 1/5 | Step 400/599 | Loss: 0.9265
Epoch 1/5 | Step 450/599 | Loss: 0.8506
Epoch 1/5 | Step 500/599 | Loss: 0.8568
Epoch 1/5 | Step 550/599 | Loss: 0.5522

Epoch 1/5 결과
Train Loss: 0.6274
Valid Accuracy: 0.7391
Valid Precision: 0.4881
Valid Recall: 0.5395
Valid F1: 0.5125
----------------------------------------------------------------------
Epoch 2/5 | Step 50/599 | Loss: 0.4984
Epoch 2/5 | Step 100/599 | Loss: 0.6199
Epoch 2/5 | Step 150/599 | Loss: 0.6219
Epoch 2/5 | Step 200/599 | Loss: 0.3891
Epoch 2/5 | Step 250/599 | Loss: 0.4857
Epoch 2/5 | Step 300/599 | Loss: 0.5772
Epoch 2/5 | Step 350/599 | Loss: 0.5531
Epoch 2/5 | Step 400/599 | Loss: 0.4796
Epoch 2/5 | S

## 12. Baseline 결과 비교표 저장

In [ ]:
result_df = pd.DataFrame([bert_result, roberta_result])

display(result_df)

result_df.to_csv(
    "baseline_result.csv",
    index=False,
    encoding="utf-8-sig"
)

print("결과 저장 완료: baseline_result.csv")

,model,accuracy,precision,recall,f1
0,BERT_comment_only_class_weight,0.596667,0.347222,0.649351,0.452489
1,RoBERTa_comment_only_class_weight,0.636667,0.370968,0.597403,0.457711


결과 저장 완료: baseline_result.csv


## 13. 생성된 파일 확인

In [ ]:
print("현재 폴더 파일 목록:")
for name in os.listdir("."):
    print(name)

현재 폴더 파일 목록:
baseline_result.csv
baseline_test.csv
baseline_train.csv
baseline_valid.csv
bert_comment_baseline
kocoh_baseline_final_v2.ipynb
KOCOH_v.2.csv
prepare_data.py
roberta_comment_baseline
team_test.csv
team_train.csv
team_valid.csv


## 14. 팀원에게 전달할 파일

팀원에게 보낼 파일:
- `team_train.csv`
- `team_valid.csv`
- `team_test.csv`
- `baseline_result.csv`

팀원은 `team_*.csv`에서 `context + comment`를 사용해서 제안 모델을 만들면 됩니다.

## 15. 보고서/발표용 문장

본 실험에서는 KOCOH 데이터셋을 사용하여 댓글 단독 기반의 혐오표현 탐지 Baseline 모델을 구축하였다.  
Baseline 모델은 맥락 정보를 제외하고 댓글 본문만 입력으로 사용하였으며, BERT와 RoBERTa 기반 이진 분류 모델을 각각 학습하였다.  
이를 통해 맥락 없이 댓글만 보았을 때의 기본 성능을 확인하고, 이후 context와 comment를 함께 사용하는 제안 모델과 성능을 비교할 기준을 마련하였다.

-아쉬운 점

BERT 및 RoBERTa 기반 댓글 단독 분류 모델을 실험한 결과,
Accuracy는 일정 수준 확보되었지만 Precision, Recall, F1-score는 매우 낮게 나타났다.

이는 댓글 본문만으로는 맥락 의존적인 혐오표현을 충분히 구분하지 못했기 때문으로 해석할 수 있다.